# Fase 5 · M06: Ensambles — VERSIÓN PRUEBA

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

## ⚡ Versión prueba

Subset estratificado 20% del train · 3-Fold CV · Sin Optuna · Sin serialización definitiva.
Objetivo: verificar que Voting, Stacking, Bagging y AdaBoost funcionan end-to-end.

## 📤 Genera

`docs/html/fase5/m06_ensambles_prueba.html`

## ➡️ Siguiente

Si todo es correcto → `f5_m06_ensambles.ipynb` con `FORZAR_OPTUNA = True`


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
import json
import time
import tracemalloc
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.ensemble import (
    VotingClassifier, StackingClassifier,
    BaggingClassifier, AdaBoostClassifier,
    RandomForestClassifier, GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, StratifiedShuffleSplit, cross_validate
)
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    accuracy_score, cohen_kappa_score, log_loss,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(6):
        if (ROOT / 'src').exists():
            break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina

RUTA_MODELADO = ROOT / 'data' / '05_modelado'
RUTA_MODELS   = RUTA_MODELADO / 'models'
RUTA_HTML_F5  = RUTA_HTML / 'fase5'
crear_directorios([RUTA_HTML_F5])

RANDOM_STATE = 42
N_SPLITS_CV  = 3
SUBSET_RATIO = 0.20
FAMILIA      = 'ensambles'
ESTRATEGIAS  = ['none', 'balanced', 'smote']
MODELOS_M06  = ['Voting', 'Stacking', 'Bagging', 'AdaBoost']
COLOR        = '#3182ce'
colores_4    = ['#3182ce', '#e53e3e', '#38a169', '#d69e2e']
fmt          = formato_numero_es

info_entorno()
print(f'\n⚡ Modo PRUEBA — subset {int(SUBSET_RATIO*100)}% · {N_SPLITS_CV}-Fold · sin Optuna')


✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGA DE DATOS + SUBSET + MODELOS BASE
# ============================================================================

X_train = pd.read_parquet(RUTA_MODELADO / 'X_train.parquet')
X_test  = pd.read_parquet(RUTA_MODELADO / 'X_test.parquet')
y_train = pd.read_parquet(RUTA_MODELADO / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(RUTA_MODELADO / 'y_test.parquet').squeeze()

pipeline_prep = joblib.load(RUTA_MODELADO / 'pipeline_preprocesamiento.pkl')

ruta_train_prep = RUTA_MODELADO / 'X_train_prep.parquet'
ruta_test_prep  = RUTA_MODELADO / 'X_test_prep.parquet'

if ruta_train_prep.exists() and ruta_test_prep.exists():
    X_train_prep = pd.read_parquet(ruta_train_prep)
    X_test_prep  = pd.read_parquet(ruta_test_prep)
    print('✅ Preprocesados cargados desde disco')
else:
    X_train_prep = pd.DataFrame(
        pipeline_prep.transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_prep = pd.DataFrame(
        pipeline_prep.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )
    print('✅ Preprocesados generados')

# Subset 20%
sss = StratifiedShuffleSplit(n_splits=1, test_size=1-SUBSET_RATIO, random_state=RANDOM_STATE)
idx_sub, _ = next(sss.split(X_train_prep, y_train))
X_train_prep = X_train_prep.iloc[idx_sub]
y_train      = y_train.iloc[idx_sub]

FEATURE_NAMES = list(X_train.columns)
N_FEATURES    = len(FEATURE_NAMES)

# Modelos base para Voting/Stacking
CANDIDATOS_BASE = [
    ('LogReg',   'LogisticRegression__smote.pkl',  LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ('RF',       'RandomForest__smote.pkl',         RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
    ('CatBoost', 'CatBoost__smote.pkl',             GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ('EBM',      'EBM__none.pkl',                   LogisticRegression(max_iter=500, random_state=RANDOM_STATE)),
]

modelos_base = []
print('\nModelos base:')
for nombre, pkl_name, fallback in CANDIDATOS_BASE:
    ruta_pkl = RUTA_MODELS / pkl_name
    if ruta_pkl.exists():
        modelo = joblib.load(ruta_pkl)
        print(f'  ✅ {nombre}: {pkl_name}')
    else:
        modelo = Pipeline([('model', fallback)])
        print(f'  ⚠️  {nombre}: fallback {fallback.__class__.__name__}')
    modelos_base.append((nombre, modelo))

print(f'\nSubset: {fmt(len(X_train_prep))} filas ({int(SUBSET_RATIO*100)}%)  abandono={y_train.mean()*100:.1f}%')
print(f'X_test : {fmt(len(X_test))} × {N_FEATURES}  abandono={y_test.mean()*100:.1f}%')


✅ Preprocesados cargados desde disco

Modelos base:
  ⚠️  LogReg: fallback LogisticRegression
  ✅ RF: RandomForest__smote.pkl
  ✅ CatBoost: CatBoost__smote.pkl
  ⚠️  EBM: fallback LogisticRegression

Subset: 5.379 filas (20%)  abandono=29.2%
X_test : 6.725 × 19  abandono=29.2%


In [3]:
# ============================================================================
# CELDA 3: FUNCIONES AUXILIARES
# ============================================================================

CV = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)


def construir_pipeline(modelo, estrategia: str):
    if estrategia == 'smote':
        return ImbPipeline([
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('model', modelo)
        ])
    return Pipeline([('model', modelo)])


def evaluar_cv(nombre: str, modelo, X, y, estrategia: str) -> dict:
    pipe = construir_pipeline(modelo, estrategia)
    scoring = {'f1': 'f1', 'roc_auc': 'roc_auc',
               'precision': 'precision', 'recall': 'recall', 'accuracy': 'accuracy'}
    tracemalloc.start()
    t0     = time.time()
    cv_res = cross_validate(pipe, X, y, cv=CV, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    t_total = time.time() - t0
    _, mem_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        'modelo': nombre, 'estrategia': estrategia, 'familia': FAMILIA,
        'f1_mean': cv_res['test_f1'].mean(), 'f1_std': cv_res['test_f1'].std(),
        'auc_mean': cv_res['test_roc_auc'].mean(), 'auc_std': cv_res['test_roc_auc'].std(),
        'precision_mean': cv_res['test_precision'].mean(),
        'recall_mean': cv_res['test_recall'].mean(),
        'accuracy_mean': cv_res['test_accuracy'].mean(),
        'tiempo_s': round(t_total, 3), 'memoria_mb': round(mem_pico / 1024**2, 2),
    }


def calcular_metricas_test(nombre: str, pipe_fit, X_te, y_te, estrategia: str) -> dict:
    y_pred  = pipe_fit.predict(X_te)
    y_proba = pipe_fit.predict_proba(X_te)[:, 1]
    return {
        'modelo': nombre, 'estrategia': estrategia,
        'f1_test'        : round(f1_score(y_te, y_pred), 4),
        'auc_test'       : round(roc_auc_score(y_te, y_proba), 4),
        'auc_pr_test'    : round(average_precision_score(y_te, y_proba), 4),
        'precision_test' : round(precision_score(y_te, y_pred), 4),
        'recall_test'    : round(recall_score(y_te, y_pred), 4),
        'accuracy_test'  : round(accuracy_score(y_te, y_pred), 4),
        'kappa_test'     : round(cohen_kappa_score(y_te, y_pred), 4),
        'logloss_test'   : round(log_loss(y_te, y_proba), 4),
    }


print('✅ Funciones definidas')


✅ Funciones definidas


In [4]:
# ============================================================================
# CELDA 4: HIPERPARÁMETROS POR DEFECTO (sin Optuna en versión prueba)
# ============================================================================

PARAMS_OPTUNA = {
    'Voting'   : {},
    'Stacking' : {},
    'Bagging'  : {'n_estimators': 50, 'max_samples': 0.8, 'max_features': 1.0},
    'AdaBoost' : {'n_estimators': 100, 'learning_rate': 0.1},
}
AUC_OPTUNA    = {m: None for m in MODELOS_M06}
mejores_params = PARAMS_OPTUNA

print('⚡ Params por defecto (versión prueba — sin Optuna)')
for nombre, params in mejores_params.items():
    print(f'   {nombre:<12}: {params if params else "defaults"}')


⚡ Params por defecto (versión prueba — sin Optuna)
   Voting      : defaults
   Stacking    : defaults
   Bagging     : {'n_estimators': 50, 'max_samples': 0.8, 'max_features': 1.0}
   AdaBoost    : {'n_estimators': 100, 'learning_rate': 0.1}


In [5]:
# ============================================================================
# CELDA 5: ENTRENAMIENTO — 12 COMBINACIONES
# ============================================================================

def construir_modelo(nombre: str, estrategia: str):
    p = mejores_params.get(nombre, {})
    if nombre == 'Voting':
        return VotingClassifier(estimators=modelos_base, voting='soft', n_jobs=-1)
    elif nombre == 'Stacking':
        return StackingClassifier(
            estimators=modelos_base,
            final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            cv=3, n_jobs=-1
        )
    elif nombre == 'Bagging':
        return BaggingClassifier(
            estimator=SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
            n_estimators=p.get('n_estimators', 50),
            max_samples=p.get('max_samples', 0.8),
            max_features=p.get('max_features', 1.0),
            random_state=RANDOM_STATE, n_jobs=-1
        )
    elif nombre == 'AdaBoost':
        return AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=3),
            n_estimators=p.get('n_estimators', 100),
            learning_rate=p.get('learning_rate', 0.1),
            random_state=RANDOM_STATE
        )


modelos_fit     = {}
resultados_cv   = []
resultados_test = []
emisiones       = None

print('=' * 60)
print('ENTRENAMIENTO PRUEBA — 4 modelos × 3 estrategias = 12 combinaciones')
print('=' * 60)
t0_total = time.time()

for nombre in MODELOS_M06:
    print(f'  {nombre}...', end=' ', flush=True)
    for estrategia in ESTRATEGIAS:
        clave  = f'{nombre}__{estrategia}'
        modelo = construir_modelo(nombre, estrategia)
        res_cv = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                            X_train_prep, y_train, estrategia)
        resultados_cv.append(res_cv)
        pipe_final = construir_pipeline(modelo, estrategia)
        pipe_final.fit(X_train_prep, y_train)
        modelos_fit[clave] = pipe_final
        res_test = calcular_metricas_test(nombre, pipe_final, X_test_prep, y_test, estrategia)
        resultados_test.append(res_test)
    print('✅')

df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
df_test = pd.DataFrame(resultados_test)
df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])

MEJOR_MODELO     = df_cv.iloc[0]['modelo']
MEJOR_ESTRATEGIA = df_cv.iloc[0]['estrategia']
mejor_clave      = f'{MEJOR_MODELO}__{MEJOR_ESTRATEGIA}'
mejor_pipe       = modelos_fit[mejor_clave]

print(f'\n⏱  Tiempo total: {time.time()-t0_total:.1f}s')
print(f'\n🏆 Mejor (orientativo): {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')


ENTRENAMIENTO PRUEBA — 4 modelos × 3 estrategias = 12 combinaciones
✅ Voting... 
✅ Stacking... 
✅ Bagging... 
✅ AdaBoost... 

⏱  Tiempo total: 1169.0s

🏆 Mejor (orientativo): Stacking + smote
   AUC CV = 0.9220 ± 0.0074
   F1  CV = 0.7699 ± 0.0078


In [6]:
# ============================================================================
# CELDA 6: TABLA RESULTADOS
# ============================================================================

print(df_cv[['modelo','estrategia','auc_mean','auc_std','f1_mean',
             'precision_mean','recall_mean','tiempo_s']]
      .to_string(index=False, float_format='{:.4f}'.format))


  modelo estrategia  auc_mean  auc_std  f1_mean  precision_mean  recall_mean  tiempo_s
Stacking      smote    0.9220   0.0074   0.7699          0.7748       0.7654  195.7810
Stacking       none    0.9213   0.0062   0.7627          0.8141       0.7177  150.2550
Stacking   balanced    0.9213   0.0062   0.7627          0.8141       0.7177   95.7310
  Voting       none    0.9136   0.0053   0.7509          0.7861       0.7190   50.9040
  Voting   balanced    0.9136   0.0053   0.7509          0.7861       0.7190   55.7440
  Voting      smote    0.9116   0.0056   0.7630          0.7403       0.7883   31.1530
AdaBoost      smote    0.9059   0.0042   0.7396          0.7008       0.7832    3.2500
AdaBoost       none    0.9046   0.0053   0.7228          0.8194       0.6472    2.3520
AdaBoost   balanced    0.9046   0.0053   0.7228          0.8194       0.6472    1.8160
 Bagging       none    0.8911   0.0054   0.7096          0.8054       0.6345   83.3480
 Bagging   balanced    0.8911   0.0054   0.

In [7]:
# ============================================================================
# CELDA 7: ANÁLISIS MODELOS BASE + GRÁFICOS
# ============================================================================

graficos_b64 = {}

# --- Modelos base vs Voting ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax_v, ax_s = axes

nombres_base = [n for n, _ in modelos_base]
aucs_base    = []
for nombre_b, pipe_b in modelos_base:
    try:
        cv_b = cross_validate(pipe_b, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=-1)
        aucs_base.append(cv_b['test_score'].mean())
    except Exception:
        aucs_base.append(0.0)

auc_voting   = df_cv[df_cv['modelo'] == 'Voting']['auc_mean'].max()
nombres_plot = nombres_base + ['Voting']
aucs_plot    = aucs_base + [auc_voting]
colores_v    = [COLOR] * len(nombres_base) + ['#e53e3e']

bars_v = ax_v.bar(nombres_plot, aucs_plot, color=colores_v, edgecolor='white')
ax_v.set_ylim(max(0.5, min(aucs_plot)-0.02), min(1.0, max(aucs_plot)+0.02))
ax_v.set_ylabel('AUC-ROC (CV)')
ax_v.set_title('Modelos base vs Voting (prueba)', fontweight='bold')
for bar, val in zip(bars_v, aucs_plot):
    ax_v.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
              f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax_v.spines['top'].set_visible(False); ax_v.spines['right'].set_visible(False)

# --- Stacking coeficientes ---
try:
    mejor_est_stack = (df_cv[df_cv['modelo']=='Stacking']
                       .sort_values('auc_mean',ascending=False).iloc[0]['estrategia'])
    pipe_stack  = modelos_fit[f'Stacking__{mejor_est_stack}']
    meta_modelo = pipe_stack.named_steps['model'].final_estimator_
    coefs       = meta_modelo.coef_[0]
    ax_s.barh(nombres_base, np.abs(coefs),
              color=[COLOR if c >= 0 else '#e53e3e' for c in coefs], height=0.5)
    ax_s.set_xlabel('|Coeficiente| meta-modelo')
    ax_s.set_title('Stacking — peso modelos base (prueba)', fontweight='bold')
except Exception as e:
    ax_s.text(0.5, 0.5, f'No disponible:\n{e}', ha='center', va='center',
              transform=ax_s.transAxes)
    ax_s.set_title('Stacking — pesos meta-modelo', fontweight='bold')

ax_s.spines['top'].set_visible(False); ax_s.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['modelos_base'] = figura_a_base64(fig)
plt.close(fig)

# --- Calibración + ROC + Comparativa AUC ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ax_cal, ax_roc, ax_bar = axes

ax_cal.plot([0,1],[0,1],'k--',linewidth=1.2,label='Perfecta')
ax_roc.plot([0,1],[0,1],'k--',linewidth=1)

aucs_bar = []
for idx, nombre_m in enumerate(MODELOS_M06):
    mejor_est = (df_cv[df_cv['modelo']==nombre_m]
                 .sort_values('auc_mean',ascending=False).iloc[0]['estrategia'])
    pipe_m  = modelos_fit[f'{nombre_m}__{mejor_est}']
    y_proba = pipe_m.predict_proba(X_test_prep)[:,1]
    color_m = colores_4[idx % len(colores_4)]

    frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)
    ax_cal.plot(mean_pred, frac_pos, 'o-', color=color_m,
                label=nombre_m, linewidth=1.8, markersize=4)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_v = roc_auc_score(y_test, y_proba)
    ax_roc.plot(fpr, tpr, color=color_m, label=f'{nombre_m} {auc_v:.3f}', linewidth=2)
    aucs_bar.append(df_cv[df_cv['modelo']==nombre_m]['auc_mean'].max())

bars = ax_bar.bar(MODELOS_M06, aucs_bar, color=colores_4, edgecolor='white')
ax_bar.set_ylim(max(0.5, min(aucs_bar)-0.02), min(1.0, max(aucs_bar)+0.02))
ax_bar.set_ylabel('AUC-ROC (CV)')
ax_bar.set_title('Comparativa AUC ensambles', fontweight='bold')
for bar, val in zip(bars, aucs_bar):
    ax_bar.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

for ax, titulo in zip([ax_cal, ax_roc, ax_bar],
                      ['Calibración','ROC comparativa','AUC comparativo']):
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax_cal.set_xlabel('Prob. predicha'); ax_cal.set_ylabel('Fracción reales')
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')

plt.tight_layout()
graficos_b64['graficos_principales'] = figura_a_base64(fig)
plt.close(fig)

# --- Confusión ---
y_pred_mejor = mejor_pipe.predict(X_test_prep)
cm = confusion_matrix(y_test, y_pred_mejor)
fig, ax_cm = plt.subplots(figsize=(6, 5))
im = ax_cm.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax_cm)
clases = ['Continúa (0)', 'Abandona (1)']
ax_cm.set_xticks([0,1]); ax_cm.set_xticklabels(clases, rotation=30, ha='right')
ax_cm.set_yticks([0,1]); ax_cm.set_yticklabels(clases)
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                   color='white' if cm[i,j] > thresh else 'black',
                   fontsize=13, fontweight='bold')
ax_cm.set_title(f'Confusión — {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}', fontweight='bold')
ax_cm.set_ylabel('Real'); ax_cm.set_xlabel('Predicho')
plt.tight_layout()
graficos_b64['cm'] = figura_a_base64(fig)
plt.close(fig)

print('✅ Gráficos generados')
print(classification_report(y_test, y_pred_mejor, target_names=['Continúa','Abandona']))


✅ Gráficos generados
              precision    recall  f1-score   support

    Continúa       0.91      0.90      0.91      4758
    Abandona       0.77      0.79      0.78      1967

    accuracy                           0.87      6725
   macro avg       0.84      0.85      0.84      6725
weighted avg       0.87      0.87      0.87      6725



In [8]:
# ============================================================================
# CELDA 8: GENERAR HTML
# ============================================================================

RUTA_HTML_SALIDA = RUTA_HTML_F5 / 'm06_ensambles_prueba.html'

aviso_prueba = '''
<div style="background:#fff3cd;border:1px solid #ffc107;border-radius:8px;
            padding:14px 18px;margin-bottom:20px;">
  <strong>⚡ Versión prueba</strong> — Subset 20% del train · 3-Fold CV · Sin Optuna<br>
  Resultados orientativos. Ejecutar <code>f5_m06_ensambles.ipynb</code> para los artefactos definitivos.
</div>'''

kpis = [
    {'valor': '4',                                  'titulo': 'Modelos'},
    {'valor': '3',                                  'titulo': 'Estrategias'},
    {'valor': '12',                                 'titulo': 'Combinaciones'},
    {'valor': '❌ Prueba',                           'titulo': 'Optuna'},
    {'valor': f'{df_cv.iloc[0]["auc_mean"]:.3f}',   'titulo': 'Mejor AUC CV'},
    {'valor': f'{df_cv.iloc[0]["f1_mean"]:.3f}',    'titulo': 'Mejor F1 CV'},
]
kpis_html = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:16px 0;">' + ''.join(
    f'<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:10px;'
    f'padding:18px 28px;text-align:center;min-width:120px;">'
    f'<div style="font-size:2rem;font-weight:700;color:#3182ce;">{k["valor"]}</div>'
    f'<div style="font-size:0.85rem;color:#555;margin-top:4px;">{k["titulo"]}</div>'
    f'</div>'
    for k in kpis
) + '</div>'

filas_cv = ''
for _, r in df_cv.iterrows():
    es_mejor = r['modelo'] == MEJOR_MODELO and r['estrategia'] == MEJOR_ESTRATEGIA
    bg = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    filas_cv += (
        f'<tr style="{bg}"><td>{r["modelo"]}</td><td>{r["estrategia"]}</td>'
        f'<td>{r["auc_mean"]:.4f} ± {r["auc_std"]:.4f}</td>'
        f'<td>{r["f1_mean"]:.4f}</td><td>{r["precision_mean"]:.4f}</td>'
        f'<td>{r["recall_mean"]:.4f}</td><td>{r["tiempo_s"]:.1f}s</td></tr>'
    )
tabla_cv = (
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Estrategia</th><th>AUC (mean±std)</th>'
    '<th>F1</th><th>Precisión</th><th>Recall</th><th>Tiempo</th>'
    f'</tr></thead><tbody>{filas_cv}</tbody></table>'
)

def img_tag(key):
    b64 = graficos_b64.get(key)
    if not b64:
        return '<p><em>Gráfico no disponible</em></p>'
    return f'<img src="data:image/png;base64,{b64}" style="max-width:100%;border-radius:8px;">'

secciones = (
    '<section class="seccion"><h2>Resumen</h2>' + aviso_prueba + kpis_html + '</section>'
    '<section class="seccion"><h2>Resultados 3-Fold CV — 12 combinaciones (orientativo)</h2>' + tabla_cv + '</section>'
    '<section class="seccion"><h2>Modelos base vs Voting · Stacking pesos</h2>' + img_tag('modelos_base') + '</section>'
    '<section class="seccion"><h2>Calibración · ROC · Comparativa AUC</h2>' + img_tag('graficos_principales') + '</section>'
    '<section class="seccion"><h2>Matriz de confusión</h2>' + img_tag('cm') + '</section>'
)

render_pagina(
    'f5_m06_ensambles_prueba.ipynb',
    secciones,
    RUTA_HTML_SALIDA,
    carpeta_notebook='fase5_modelado'
)
print(f'\n✅ HTML generado: {RUTA_HTML_SALIDA}')



✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase5\m06_ensambles_prueba.html


In [9]:
# ============================================================================
# CELDA 9: RESUMEN FINAL
# ============================================================================

print('=' * 60)
print('RESUMEN FINAL — f5_m06_ensambles_prueba')
print('=' * 60)
print(f'Subset      : {fmt(len(X_train_prep))} filas ({int(SUBSET_RATIO*100)}% train)')
print(f'Modelos     : Voting · Stacking · Bagging · AdaBoost')
print(f'Estrategias : none · balanced · smote  (12 combinaciones)')
print(f'CV folds    : {N_SPLITS_CV}  (reducido)')
print( 'Optuna      : ❌ No ejecutado (params por defecto)')
print(f'Modelos base: {[n for n, _ in modelos_base]}')
print()
print(f'🏆 Mejor (orientativo): {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')
print()
print('✅ Pipeline verificado — listo para lanzar f5_m06_ensambles.ipynb')


RESUMEN FINAL — f5_m06_ensambles_prueba
Subset      : 5.379 filas (20% train)
Modelos     : Voting · Stacking · Bagging · AdaBoost
Estrategias : none · balanced · smote  (12 combinaciones)
CV folds    : 3  (reducido)
Optuna      : ❌ No ejecutado (params por defecto)
Modelos base: ['LogReg', 'RF', 'CatBoost', 'EBM']

🏆 Mejor (orientativo): Stacking + smote
   AUC CV = 0.9220 ± 0.0074
   F1  CV = 0.7699 ± 0.0078

✅ Pipeline verificado — listo para lanzar f5_m06_ensambles.ipynb
